# Audit — GPU checks: DeepSeek precision rung, Gemma device split, GPU decode vs prefill

**Settings:** Accelerator **GPU T4 ×2** · Internet **On** · Secrets `GITHUB_TOKEN` (if private), `HF_TOKEN`.
**Add Input:**
* the output of `audit_convert_deepseek_cpu` (`deepseek-v2-lite-Q8_0.gguf`, `convert_report.json`)
* models `ryzewtf/moe-panel/gguf/deepseek-v2-lite`, `.../gemma-4-26b-a4b`, `.../olmoe-0125-instruct`,
  `.../olmoe-0125-f16`
* dataset `moe-corpus-v2` (`mixed-v2.jsonl`)

Run All. Roughly: build ~25 min; DeepSeek Q8_0 + Q4_K_M on 2 shards ~15 min; Gemma two splits on 2
shards ~20 min; OLMoE-Instruct prefill + decode on 1 shard ~10 + 30–60 min; optional F16 decode
another 30–60 min (skipped automatically if under `RESERVE_H` hours remain). Every rung is saved to
`/kaggle/working/audit_gpu/` as soon as it finishes; the comparisons are in `reports/`.


In [ ]:
# ============================================================================
# PARAMETERS
# ============================================================================
GIT_REF = "VLLM_PORT"
CORPUS_NAME = "mixed-v2"
SHARDS = "0-1"            # ~150k tokens, the Phase 8 ladder's shards
DECODE_SHARDS = "0"       # one shard (~76k tokens) for the slow one-token-at-a-time legs
ALT_SPLIT = [0.35, 0.65]  # T8.5 candidate; the pinned collection split is [0.5, 0.5]
RUN_F16_DECODE = True     # OLMoE F16 prefill vs decode (kernel/quantization question)
SESSION_H = 12.0
RESERVE_H = 1.5           # skip optional legs when less than this is left
WORK = "/tmp/audit"       # /tmp is the ~1 TB scratch disk; /kaggle/temp does not exist on Kaggle


In [ ]:
# ============================================================================
# Bootstrap: clone the repo at GIT_REF, install what the image lacks.
# ============================================================================
import os, subprocess, sys, time, shutil
from pathlib import Path

GIT_URL = "https://github.com/ryzewtf/GenAI-IA-1.git"
REPO = Path("/kaggle/working/repo")
T0 = time.time()


def run(cmd, cwd=None, check=True, quiet=False):
    """Run a command, stream it, and fail loudly.

    `check=True` by default on purpose: a step that fails and lets the notebook carry on
    produces a session whose later cells operate on the previous step's leftovers, which is
    the expensive kind of wrong here -- it looks like it worked.
    """
    if not quiet:
        print("$", " ".join(str(c) for c in cmd), flush=True)
    p = subprocess.run([str(c) for c in cmd], cwd=cwd and str(cwd), text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if p.stdout and not quiet:
        print(p.stdout, flush=True)
    if check and p.returncode != 0:
        raise SystemExit(f"FAILED ({p.returncode}): {' '.join(str(c) for c in cmd)}")
    return p


# A private repo needs a token; a public one must not be handed one. Kaggle Secrets is the only
# place a token belongs -- an inline PAT is committed the moment the notebook is saved.
url = GIT_URL
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if tok:
        url = GIT_URL.replace("https://", f"https://{tok}@")
        print("using GITHUB_TOKEN from Kaggle Secrets")
except Exception:
    print("no GITHUB_TOKEN secret; cloning anonymously (fine if the repo is public)")

if REPO.exists():
    run(["git", "fetch", "--all", "--tags"], cwd=REPO)
    run(["git", "checkout", GIT_REF], cwd=REPO)
    run(["git", "pull", "--ff-only"], cwd=REPO, check=False)
else:
    run(["git", "clone", url, str(REPO)])
    run(["git", "checkout", GIT_REF], cwd=REPO)

HEAD = run(["git", "rev-parse", "HEAD"], cwd=REPO, quiet=True).stdout.strip()
print(f"\nrepo at {HEAD}  (ref {GIT_REF})")

os.chdir(REPO)
sys.path.insert(0, str(REPO))

# --no-deps: the Kaggle image pins torch/numpy against each other, and letting pip resolve
# transitive deps here is how a session ends up with a numpy the image's torch was not built
# against. Same rule as PIP_PACKAGES in src/runtime/setup_kaggle.py.
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
     "huggingface_hub", "datasets", "sentencepiece", "gguf", "kagglehub"], check=False)

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")   # xet hangs at 0%/99% in managed notebooks
os.environ.setdefault("PYTHONUNBUFFERED", "1")

# The multilingual domain's parallel control (facebook/flores) and the code substitute
# (starcoderdata) are gated on the HF Hub: without a token the corpus fetch dies with
# DatasetNotFoundError partway through. The token lives in Kaggle Secrets, never inline --
# an HF_TOKEN pasted into a cell is leaked the moment the notebook is saved. Both env names
# are set because huggingface_hub reads HF_TOKEN and older datasets code reads the other.
try:
    from kaggle_secrets import UserSecretsClient
    _hf = UserSecretsClient().get_secret("HF_TOKEN")
    if _hf:
        os.environ["HF_TOKEN"] = _hf
        os.environ["HUGGINGFACE_HUB_TOKEN"] = _hf
        print("using HF_TOKEN from Kaggle Secrets")
except Exception:
    print("no HF_TOKEN secret; gated datasets (flores, starcoderdata) will 404 in a corpus run")


In [ ]:
# ============================================================================
# Locate every input BEFORE the build: a missing attachment costs seconds, not 25 minutes.
# ============================================================================
import json as _json, glob

# Hardware first: the Gemma split and the 15.6 GiB Q8_0 file both need two T4s (sm_75, as built).
_gpus = run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            check=False, quiet=True).stdout.strip().splitlines()
print("GPUs:", _gpus)
if len(_gpus) != 2 or not all("T4" in g for g in _gpus):
    raise SystemExit(f"need Accelerator 'GPU T4 x2', found {_gpus or 'no GPU'}")
if shutil.which("nvcc") is None and not Path("/usr/local/cuda/bin/nvcc").exists():
    raise SystemExit("nvcc not found: this image cannot build moe_trace")
Path(WORK).mkdir(parents=True, exist_ok=True)
_free = shutil.disk_usage(WORK).free / 2**30
print(f"{WORK}: {_free:.0f} GiB free")
if _free < 25:
    raise SystemExit(f"{WORK} has {_free:.0f} GiB free; the rungs need ~10 GiB plus the build")

def _one(pattern):
    hits = sorted(Path("/kaggle/input").rglob(pattern))
    if len(hits) != 1:
        raise SystemExit(f"expected exactly one {pattern} under /kaggle/input, found {hits}")
    return hits[0]

DS_Q8 = _one("deepseek-v2-lite-Q8_0.gguf")
DS_Q4 = _one("deepseek-v2-lite-Q4_K_M.gguf")
GEMMA = _one("gemma-4-26b-a4b-Q4_K_M.gguf")
OLMI_Q4 = _one("olmoe-0125-instruct-Q4_K_M.gguf")
OLM_F16 = _one("olmoe-0125-F16.gguf")
CORPUS = _one("mixed-v2.jsonl")
rep = _json.loads(_one("convert_report.json").read_text())
if not (rep.get("q4_identical_to_panel") and rep.get("q8_router_audit_pass")):
    raise SystemExit(f"convert_report.json does not clear the Q8_0 rung: {rep}")
with open(CORPUS, encoding="utf-8") as fh:
    for i, line in enumerate(fh):
        if line.strip() and _json.loads(line)["doc_id"] != i:
            raise SystemExit(f"{CORPUS} line {i}: not the renumbered mixed-v2 corpus")
for p in (DS_Q8, DS_Q4, GEMMA, OLMI_Q4, OLM_F16):
    print(f"{p.name:36s} {p.stat().st_size/2**30:5.1f} GiB  {p}")
print("corpus", CORPUS, f"{i+1} docs OK; Q8_0 sha {rep['q8_sha256'][:12]}")


In [ ]:
# ============================================================================
# Build moe_trace (CUDA sm_75) with the T1.2/T1.4 gates on DeepSeek; router audit on the Q8_0 file.
# The node specs for Gemma and OLMoE are committed and were gated on these exact files earlier.
# ============================================================================
run([sys.executable, "scripts/kaggle_setup.py", "--model", "deepseek-v2-lite"])
MOE_TRACE = next(p for root in ("/tmp", "/kaggle/temp") for p in Path(root).rglob("moe_trace")
                 if p.is_file() and "CMakeFiles" not in str(p))
print("moe_trace", MOE_TRACE)
run([sys.executable, "-m", "src.capture.router_audit", str(DS_Q8),
     "-o", "results/router_dtype_audit_ds_q8.csv"])


In [ ]:
# ============================================================================
# Rungs. Own scratch (=> own shard ledger) per rung; --quant per rung; each saved the moment it exists.
# ============================================================================
import re, zipfile
OUT = Path("/kaggle/working/audit_gpu"); (OUT / "reports").mkdir(parents=True, exist_ok=True)
RUN_YAML = Path("configs/run.yaml")
RUN_YAML_TEXT = RUN_YAML.read_text()
assert RUN_YAML_TEXT.count("tensor_split: [0.5, 0.5]") == 1, "run.yaml no longer pins tensor_split [0.5, 0.5] once"
DONE, FAILED = {}, []

def hours_left():
    return SESSION_H - (time.time() - T0) / 3600

def root_of(name):
    return next((Path(WORK) / name / "out").rglob("shard_00000")).parents[2]

def save_slim(name):
    # topk/logits/tokens/manifests only: the comparisons never read hidden.bin, and it is most of the bytes
    root = root_of(name)
    with zipfile.ZipFile(OUT / f"trace_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for f in root.rglob("*"):
            if f.is_file() and f.name != "hidden.bin":
                z.write(f, f.relative_to(root.parent))

def rung(name, model, gguf, quant, shards, *, split=None, decode="off", timeout_s=None):
    val = Path(f"results/validate_{model}_{CORPUS_NAME}.json")
    val.unlink(missing_ok=True)   # a failed rung must not inherit the previous rung's report
    if split is not None:
        RUN_YAML.write_text(RUN_YAML_TEXT.replace("tensor_split: [0.5, 0.5]",
                                                  f"tensor_split: [{split[0]}, {split[1]}]"))
    try:
        cmd = [sys.executable, "scripts/kaggle_collect.py", "--model", model,
               "--corpus", str(CORPUS), "--corpus-name", CORPUS_NAME,
               "--gguf", str(gguf), "--quant", quant, "--binary", str(MOE_TRACE),
               "--variant-name", name, "--shards", shards, "--decode-mode", decode,
               "--backend", "local", "--scratch", str(Path(WORK) / name),
               "--local-root", str(Path(WORK) / name / "out"), "--preflight-gb", "0"]
        if timeout_s:
            cmd += ["--timeout-s", str(timeout_s)]
        rc = run(cmd, check=False).returncode
    finally:
        RUN_YAML.write_text(RUN_YAML_TEXT)   # never leave the pinned split edited
    # kaggle_collect writes results/validate_<model>_<corpus>.json, so a second rung of the same
    # model would overwrite the first: keep a per-rung copy.
    if val.exists():
        shutil.copy(val, OUT / "reports" / f"validate_{name}.json")
    try:
        m = _json.loads(next((Path(WORK) / name / "out").rglob("manifest.json")).read_text())
        save_slim(name)
        ok = rc == 0 and m["quant"] == quant
        print(f"== {name}: rc={rc} quant={m['quant']} split={m.get('device_plan', {}).get('tensor_split')} "
              f"decode={m.get('capture_variant', {}).get('decode_mode')} {'OK' if ok else 'PROBLEM'}")
    except StopIteration:
        ok = False
        print(f"== {name}: rc={rc}, NO TRACE WRITTEN")
    if ok:
        DONE[name] = model
    else:
        FAILED.append(name)
    return ok

rung("ds_q8", "deepseek-v2-lite", DS_Q8, "Q8_0", SHARDS)
rung("ds_q4", "deepseek-v2-lite", DS_Q4, "Q4_K_M", SHARDS)
rung("gemma_base", "gemma-4-26b-a4b", GEMMA, "Q4_K_M", SHARDS)
rung("gemma_split", "gemma-4-26b-a4b", GEMMA, "Q4_K_M", SHARDS, split=ALT_SPLIT)
rung("olmi_prefill", "olmoe-0125-instruct", OLMI_Q4, "Q4_K_M", DECODE_SHARDS)
rung("olmi_decode", "olmoe-0125-instruct", OLMI_Q4, "Q4_K_M", DECODE_SHARDS, decode="full",
     timeout_s=int(max(0.5, hours_left() - RESERVE_H) * 3600))
if RUN_F16_DECODE and hours_left() > RESERVE_H + 1.0:
    rung("olm16_prefill", "olmoe-0125", OLM_F16, "F16", DECODE_SHARDS)
    rung("olm16_decode", "olmoe-0125", OLM_F16, "F16", DECODE_SHARDS, decode="full",
         timeout_s=int(max(0.5, hours_left() - 0.5) * 3600))
else:
    print("F16 decode legs skipped:", f"{hours_left():.1f} h left")
print("\nrungs ok:", sorted(DONE), " failed:", FAILED or "none")


In [ ]:
# ============================================================================
# Comparisons, all against a same-session reference. None of them can stop the packaging cell.
# ============================================================================
def compare(ref, cand, mode, report, extra=()):
    if ref not in DONE or cand not in DONE:
        print(f"skip {report}: missing {ref if ref not in DONE else cand}")
        return
    run([sys.executable, "-m", "src.analysis.precision",
         "--reference-root", str(root_of(ref)), "--candidate-root", str(root_of(cand)),
         "--model", DONE[ref], "--corpus", CORPUS_NAME, "--mode", mode,
         "--report", str(OUT / "reports" / report), *extra], check=False)

compare("ds_q8", "ds_q4", "ladder", "deepseek_q8_vs_q4.json")
compare("gemma_base", "gemma_split", "T8.5", "gemma_split_T8.5.json")
compare("olmi_prefill", "olmi_decode", "T3.8", "olmoe_instruct_q4_decode_T3.8.json")
compare("olm16_prefill", "olm16_decode", "T3.8", "olmoe_f16_decode_T3.8.json")


In [ ]:
# ============================================================================
# Package: reports, router audits, validation and collection logs (traces are already zipped).
# ============================================================================
for f in glob.glob("results/router_dtype_audit_*.csv") + glob.glob("results/validate_*.json") + \
         ["results/collection_log.csv"]:
    if Path(f).exists():
        shutil.copy(f, OUT / "reports")
(OUT / "reports" / "session.json").write_text(_json.dumps(
    {"done": DONE, "failed": FAILED, "hours": (time.time() - T0) / 3600, "git_head": HEAD}, indent=2))
shutil.make_archive("/kaggle/working/audit_gpu_reports", "zip", OUT / "reports")
for f in sorted(OUT.glob("*.zip")) + [Path("/kaggle/working/audit_gpu_reports.zip")]:
    print(f"{f.name:40s} {f.stat().st_size/2**20:8.0f} MiB")
# The clone must not be saved as output: listing it pages through thousands of files (HTTP 429).
os.chdir("/kaggle/working"); shutil.rmtree(REPO, ignore_errors=True)
print(sorted(p.name for p in Path("/kaggle/working").iterdir()))
print(f"done in {(time.time()-T0)/60:.1f} min")
